# What a fine-tune learned and what it lost: style, the golden set, and the export check

Serves the base model and a LoRA adapter side by side in vLLM (the serving setup that passed the GPU risk tests)
and measures:

1. **Style** on 200 held-out prompts (100 validation, 100 from intents never trained on): the base model and
   the fine-tune, next to the dataset's own replies (`evals/style_metrics.py`). The same replies feed the
   blind side-by-side, which you fill in on the laptop (`evals/blind_compare.py`).
2. **The golden set with tools:** all 100 cases through the agent loop with exactly the baseline's settings
   (prompt v0, six tools, temperature 0.2, 4 calls per turn), for the fine-tune and, on the same server, the
   base model, so the comparison does not mix serving stacks.
3. **Export check:** 10 fixed prompts greedily through vLLM and through Transformers in this notebook (adapter
   merged into the base). Same kind of answer, or a template / end-of-turn problem.

Loss on the validation split and the held-out intents is already in the training run's `run_info.json`.

**Before running (Kaggle):** GPU **T4 x2** (vLLM on GPU 1, Transformers on GPU 0), Internet **On**; secrets
**`HF_TOKEN`** and **`NEON_URL`** switched on. Set `RUN_DATE` and `ADAPTER` in cell 1. **Save & Run All** is fine;
expect about 1.5 hours, most of it the two golden runs.

**After:** Output → download **`sft-eval.zip`** and unzip it into the repo's `evals/` folder (`runs/` and `style/`).

In [ ]:
# 1. Settings and secrets.
RUN_DATE = "2026-11-23"                              # today's date, written by hand
ADAPTER = "123jagjot/sft1-r16-lr2e-4-bitext_v1"      # the adapter on the Hub
LORA_NAME = "nova-sft1"                              # the model name vLLM serves it under
MODEL = "Qwen/Qwen3.5-2B"
SFT_RUN, BASE_RUN = f"{RUN_DATE}_sft1", f"{RUN_DATE}_floor-vllm"

import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["DB_URL"] = secrets.get_secret("NEON_URL")
print("DB host:", os.environ["DB_URL"].split("@")[-1].split("/")[0])

In [ ]:
# 2. Clone the repo; install the training stack (for the in-notebook side of the export check) and the eval client.
import subprocess, sys
REPO_DIR = os.path.abspath("customer-support-SLM")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/jagjotsingh244-source/customer-support-SLM.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
%pip install -q -r requirements-gpu.txt
%pip install -q openai "psycopg[binary]" pydantic
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
!python evals/validate.py     # the golden set must match the database the eval will query

In [ ]:
# 3. vLLM in its own environment, pinned to the versions that passed the risk tests.
!uv --version || pip install -q uv
!uv venv -q /tmp/vllm-env --python 3.12
!uv pip install -q --python /tmp/vllm-env/bin/python -r requirements-serve.txt

In [ ]:
# 4. Download the adapter, then start vLLM on GPU 1 with the base model and the adapter as a LoRA module.
import json, time, urllib.request
from huggingface_hub import snapshot_download
ADAPTER_DIR = snapshot_download(ADAPTER, local_dir="/kaggle/working/adapter",
                                allow_patterns=["adapter_model.safetensors", "adapter_config.json"])
RANK = json.load(open(f"{ADAPTER_DIR}/adapter_config.json"))["r"]
FLAGS = ("--dtype half --max-model-len 8192 --language-model-only "
         "--enable-auto-tool-choice --tool-call-parser qwen3_coder --port 8000 "
         f"--enable-lora --lora-modules {LORA_NAME}={ADAPTER_DIR} --max-lora-rank {max(RANK, 16)}")
log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen(["bash", "-c", f"source serving/t4_libcuda_fix.sh && "
                           f"exec /tmp/vllm-env/bin/vllm serve {MODEL} {FLAGS}"],
                          stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
                          env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"))
t0 = time.time()
while True:
    try:
        served = json.load(urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5))
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 1500:
            print(open("/kaggle/working/vllm.log").read()[-4000:])
            raise RuntimeError("vLLM did not start; full log in /kaggle/working/vllm.log")
        time.sleep(10)
print(f"vLLM up after {time.time() - t0:.0f}s, rank {RANK}, serving", [m["id"] for m in served["data"]])

In [ ]:
# 5. Style: 200 held-out prompts, base and fine-tune, next to the dataset's own replies.
URL = "http://localhost:8000/v1"
!python evals/generate_replies.py generate --base-url {URL} --model {MODEL} --out evals/style/base.jsonl
!python evals/generate_replies.py generate --base-url {URL} --model {LORA_NAME} --out evals/style/sft1.jsonl
!python evals/style_metrics.py --reference evals/style/base.jsonl evals/style/sft1.jsonl | tee evals/style/style-table.md
!python evals/style_metrics.py --split test_unseen_intents --reference evals/style/base.jsonl evals/style/sft1.jsonl

In [ ]:
# 6. The golden set with tools, baseline settings: the fine-tune, then the base model on the same server.
#    Re-running this cell resumes where it stopped.
!python evals/run_eval.py --run {SFT_RUN} --system candidate --provider openai --base-url {URL} \
    --model {LORA_NAME} --temperature 0.2 --notes "Kaggle T4, vLLM 0.34.0, LoRA module {ADAPTER}"
!python evals/run_eval.py --run {BASE_RUN} --system floor --provider openai --base-url {URL} \
    --model {MODEL} --temperature 0.2 --notes "Kaggle T4, vLLM 0.34.0, same server as {SFT_RUN}"

In [ ]:
# 7. Side by side with the scored floor and ceiling, and the cases that changed.
!python evals/compare.py evals/runs/2026-11-22_floor evals/runs/{BASE_RUN} evals/runs/{SFT_RUN} evals/runs/2026-11-22_ceiling
score = lambda run: json.load(open(f"evals/runs/{run}/scores.json"))["scores"]
base, sft = score(BASE_RUN), score(SFT_RUN)
broke = sorted(i for i in sft if base[i]["all_required"] and not sft[i]["all_required"])
fixed = sorted(i for i in sft if sft[i]["all_required"] and not base[i]["all_required"])
print(f"new failures (passed on the base, fail on the fine-tune): {len(broke)}\n  {' '.join(broke)}")
print(f"new passes (failed on the base, pass on the fine-tune): {len(fixed)}\n  {' '.join(fixed)}")

In [ ]:
# 8. Export check: the same 10 prompts through vLLM and through Transformers here (adapter merged, GPU 0).
!python evals/generate_replies.py generate --base-url {URL} --model {LORA_NAME} --set export \
    --out evals/style/sft1-vllm-export.jsonl
!CUDA_VISIBLE_DEVICES=0 python evals/generate_replies.py generate --provider local --model {MODEL} \
    --adapter {ADAPTER_DIR} --set export --out evals/style/sft1-notebook-export.jsonl
!python evals/generate_replies.py compare evals/style/sft1-notebook-export.jsonl evals/style/sft1-vllm-export.jsonl

In [ ]:
# 9. Package the results: Output -> sft-eval.zip, unzip into the repo's evals/ folder.
server.terminate()
!cd evals && zip -qr /kaggle/working/sft-eval.zip runs/{SFT_RUN} runs/{BASE_RUN} style
!ls -la /kaggle/working/sft-eval.zip